# Day 4 · GitHub·LangGraph 승인 Workflow

하나의 입력을 8개 차시 동안 확장합니다. 웹사이트 확인이 아니라 코드·명령·test·결과 파일을 직접 다루며, 모든 외부 쓰기는 dry-run과 사람 승인을 먼저 거칩니다.

In [1]:
from pathlib import Path
import importlib.util, json, subprocess, sys

def find_workspace(start):
    for candidate in [start, *start.parents]:
        if (candidate / "requirements-day1.txt").exists() and (candidate / "src").exists():
            return candidate
    raise RuntimeError("WORKSPACE_ROOT_NOT_FOUND")

ROOT = find_workspace(Path.cwd().resolve())
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))
print({"workspace": ROOT.name, "python": sys.version.split()[0]})

{'workspace': 'llm-agent-and-workflow-automation', 'python': '3.12.12'}


In [2]:
# 최초 1회. 없는 핵심 library가 있을 때만 현재 Notebook Kernel에 설치합니다.
required = ["pydantic", "pytest", "langchain_core", "langgraph"]
missing = [name for name in required if importlib.util.find_spec(name) is None]
if missing:
    subprocess.run(
        [sys.executable, "-m", "pip", "install", "-q", "-r", str(ROOT / "requirements-day1.txt")],
        check=True,
    )
print({"missing_before_install": missing, "environment_ready": True})

# 실제 STT를 실행할 사람만 requirements-stt-optional.txt를 별도로 설치합니다.

{'missing_before_install': [], 'environment_ready': True}


In [3]:
OUT = ROOT / "output/course-labs/day4"
OUT.mkdir(parents=True, exist_ok=True)

def save_json(name, payload):
    path = OUT / name
    path.write_text(json.dumps(payload, ensure_ascii=False, indent=2) + "\n", encoding="utf-8")
    print({"saved": str(path.relative_to(ROOT))})
    return path

def run_command(*args, cwd=ROOT):
    completed = subprocess.run(args, cwd=cwd, text=True, capture_output=True)
    display_args = list(args)
    if display_args and display_args[0] == sys.executable:
        display_args[0] = "python"
    result = {
        "command": " ".join(display_args),
        "returncode": completed.returncode,
        "stdout_tail": completed.stdout.strip().splitlines()[-5:],
        "stderr_tail": completed.stderr.strip().splitlines()[-5:],
    }
    print(json.dumps(result, ensure_ascii=False, indent=2))
    return result

## 1차시 · PR Target과 Fixture

repository·PR number·head SHA를 고정해 잘못된 대상에 게시하는 위험을 먼저 차단합니다.

In [4]:
from src.course_services.github_service import load_pr_fixture

fixture = load_pr_fixture(ROOT / "data/day4_github/pr_fixture.json", workspace_root=ROOT)
target = {key: fixture[key] for key in ("repository", "number", "head_sha")}
save_json("01_pr_target.json", target)
target

{'saved': 'output/course-labs/day4/01_pr_target.json'}


{'repository': 'classroom-owner/agent-sandbox',
 'number': 17,
 'head_sha': '7c83e8fce9a24d90b6d4a6c9123aa90817b65b1a'}

## 2차시 · GitHub 인증과 최소 권한

token 값은 읽거나 출력하지 않습니다. `.env` 제외 여부와 key 설정 유무만 확인합니다.

In [5]:
import os

ignore_check = run_command("git", "check-ignore", "-v", ".env")
auth_status = {
    "env_ignored": ignore_check["returncode"] == 0,
    "github_token_configured": bool(os.getenv("GITHUB_TOKEN")),
    "token_value_printed": False,
    "recommended_scope": "read-only until explicit sandbox approval",
}
save_json("02_auth_status.json", auth_status)
auth_status

{
  "command": "git check-ignore -v .env",
  "returncode": 0,
  "stdout_tail": [
    ".gitignore:8:.env\t.env"
  ],
  "stderr_tail": []
}
{'saved': 'output/course-labs/day4/02_auth_status.json'}


{'env_ignored': True,
 'github_token_configured': False,
 'token_value_printed': False,
 'recommended_scope': 'read-only until explicit sandbox approval'}

## 3차시 · REST 상태 코드와 복구 계약

401·403·404·422는 멈추고 원인을 고치며, 429만 제한된 횟수 안에서 재시도합니다.

In [6]:
from src.course_services.github_service import classify_github_response

response_contracts = {
    str(status): classify_github_response(status, retry_after_seconds=3)
    for status in (200, 401, 403, 404, 422, 429, 500)
}
assert response_contracts["429"]["retryable"] is True
assert all(not response_contracts[str(status)]["retryable"] for status in (401, 403, 404, 422, 500))
save_json("03_github_response_contracts.json", response_contracts)
response_contracts

{'saved': 'output/course-labs/day4/03_github_response_contracts.json'}


{'200': {'status': 'SUCCESS',
  'status_code': 200,
  'action': 'USE_RESPONSE',
  'retryable': False,
  'retry_after_seconds': None,
  'error_code': None},
 '401': {'status': 'EXPECTED_FAILURE',
  'status_code': 401,
  'action': 'STOP_AND_REAUTHENTICATE',
  'retryable': False,
  'retry_after_seconds': None,
  'error_code': 'GITHUB_AUTH_REQUIRED'},
 '403': {'status': 'EXPECTED_FAILURE',
  'status_code': 403,
  'action': 'STOP_AND_CHECK_PERMISSION',
  'retryable': False,
  'retry_after_seconds': None,
  'error_code': 'GITHUB_PERMISSION_DENIED'},
 '404': {'status': 'EXPECTED_FAILURE',
  'status_code': 404,
  'action': 'STOP_AND_CHECK_TARGET',
  'retryable': False,
  'retry_after_seconds': None,
  'error_code': 'GITHUB_TARGET_NOT_FOUND'},
 '422': {'status': 'EXPECTED_FAILURE',
  'status_code': 422,
  'action': 'REFRESH_TARGET_AND_DIFF',
  'retryable': False,
  'retry_after_seconds': None,
  'error_code': 'GITHUB_VALIDATION_FAILED'},
 '429': {'status': 'EXPECTED_FAILURE',
  'status_code': 4

## 4차시 · LangGraph State와 분기

approve·edit·reject가 같은 thread에서 서로 다른 terminal state로 끝나는지 실행합니다.

In [7]:
from src.langchain_lab import fixture_payload
from src.langgraph_lab import run_langgraph_lab

draft = fixture_payload()
graph_results = {
    decision: run_langgraph_lab(draft, decision=decision, request_id=f"day4-{decision}")
    for decision in ("approve", "edit", "reject")
}
terminal_states = {key: value["final_state"]["status"] for key, value in graph_results.items()}
assert terminal_states["reject"] == "REJECTED"
save_json("04_graph_states.json", graph_results)
terminal_states

{'saved': 'output/course-labs/day4/04_graph_states.json'}


{'approve': 'READY_FOR_EXPORT', 'edit': 'REJECTED', 'reject': 'REJECTED'}

## 5차시 · Checkpoint와 Idempotency

같은 target·SHA·comment body를 두 번 실행해 fake publisher가 한 번만 호출되는지 확인합니다.

In [8]:
from src.course_services.github_service import InMemoryIdempotencyStore, prepare_review_comment, publish_review_comment
from src.course_services.review_service import run_review_service

diff_text = (ROOT / fixture["diff"]).read_text(encoding="utf-8")
report = run_review_service(diff_text)
approval_plan = prepare_review_comment(report=report, target=target, dry_run=False)
store = InMemoryIdempotencyStore()
calls = []
def fake_publisher(pr_target, body):
    calls.append({"target": pr_target.model_dump(), "body_length": len(body)})
    return {"id": 101, "url": "https://example.invalid/reviews/101"}
first = publish_review_comment(plan=approval_plan, human_approved=True, publisher=fake_publisher, store=store)
second = publish_review_comment(plan=approval_plan, human_approved=True, publisher=fake_publisher, store=store)
idempotency_result = {"first": first, "second": second, "call_count": len(calls)}
assert len(calls) == 1 and second["remote_result"]["reused"] is True
save_json("05_idempotency.json", idempotency_result)
{"call_count": len(calls), "duplicate_reused": second["remote_result"]["reused"]}

{'saved': 'output/course-labs/day4/05_idempotency.json'}


{'call_count': 1, 'duplicate_reused': True}

## 6차시 · Human Approval과 Interrupt

사람 거절은 외부 쓰기 없이 BLOCKED로 끝나고, 승인도 target·근거·초안을 본 뒤에만 재개합니다.

In [9]:
rejected = publish_review_comment(
    plan=approval_plan, human_approved=False, publisher=fake_publisher, store=InMemoryIdempotencyStore(),
)
human_decisions = {
    "approve": {"status": first["status"], "external_write": first["external_write"], "publisher_mode": "fake"},
    "reject": rejected,
}
assert rejected["error_code"] == "HUMAN_APPROVAL_REQUIRED"
save_json("06_review_decision.json", human_decisions)
human_decisions

{'saved': 'output/course-labs/day4/06_review_decision.json'}


{'approve': {'status': 'PUBLISHED',
  'external_write': True,
  'publisher_mode': 'fake'},
 'reject': {'status': 'BLOCKED',
  'target': {'repository': 'classroom-owner/agent-sandbox',
   'number': 17,
   'head_sha': '7c83e8fce9a24d90b6d4a6c9123aa90817b65b1a'},
  'body': '## 교육용 코드 리뷰 초안\n\n- **P0 · 검증되지 않은 문자열이 코드로 실행될 수 있음**\n  - 위치: `src/payment_job.py:9`\n  - 근거: `result = eval(command)`\n  - 제안: 허용된 명령이나 파서만 호출하는 명시적 registry로 교체하세요.\n- **P1 · 외부 쓰기 전에 사람 승인 경계가 보이지 않음**\n  - 위치: `src/payment_job.py:11`\n  - 근거: `requests.post("https://example.invalid/jobs", json=payload)`\n  - 제안: dry-run payload, 대상 확인, human_approved 조건을 쓰기 호출 앞에 두세요.\n- **P2 · 넓은 예외 처리가 실패 계약을 지움**\n  - 위치: `src/payment_job.py:12`\n  - 근거: `except Exception:`\n  - 제안: 복구 가능한 예외만 잡고 구조화된 error_code를 반환하세요.\n\n자동 게시하지 않았습니다. 대상과 내용을 사람이 확인해야 합니다.',
  'idempotency_key': 'c97ba39c959d7fda585b86c3',
  'human_approval_required': True,
  'external_write': False,
  'error_code': 'HUMAN_APPROVAL_REQUIRED',
  'remote_res

## 7차시 · Dry-run Comment Payload

실제 API와 같은 body·target·idempotency key를 만들되 publisher는 호출하지 않습니다.

In [10]:
dry_run_plan = prepare_review_comment(report=report, target=target, dry_run=True)
blocked_dry_run = publish_review_comment(
    plan=dry_run_plan, human_approved=True, publisher=fake_publisher, store=InMemoryIdempotencyStore(),
)
assert dry_run_plan["external_write"] is False
assert blocked_dry_run["error_code"] == "DRY_RUN_CANNOT_PUBLISH"
save_json("07_review_comment_plan.json", {"plan": dry_run_plan, "publish_attempt": blocked_dry_run})
{"status": dry_run_plan["status"], "publish_attempt": blocked_dry_run["error_code"]}

{'saved': 'output/course-labs/day4/07_review_comment_plan.json'}


{'status': 'DRY_RUN', 'publish_attempt': 'DRY_RUN_CANNOT_PUBLISH'}

## 8차시 · Sandbox 게시 전 Audit

수업 기본 경로는 fake publisher입니다. 실제 GitHub 게시에는 본인 sandbox·최소 권한·한 건 제한·rollback이 추가로 필요합니다.

In [11]:
from src.course_services.course_demo import build_course_demo

day4_audit = build_course_demo(4, workspace_root=ROOT)
focused_test = run_command(sys.executable, "-m", "pytest", "-q", "tests/test_course_services.py", "-k", "github or dry_run or idempotent")
day4_audit["focused_test"] = focused_test
assert day4_audit["external_write"] is False
assert focused_test["returncode"] == 0
save_json("08_day4_audit_record.json", day4_audit)
{"decision": day4_audit["decision"], "metrics": day4_audit["metrics"]}

{
  "command": "python -m pytest -q tests/test_course_services.py -k github or dry_run or idempotent",
  "returncode": 0,
  "stdout_tail": [
    "...                                                                      [100%]",
    "3 passed, 13 deselected in 0.01s"
  ],
  "stderr_tail": []
}
{'saved': 'output/course-labs/day4/08_day4_audit_record.json'}


{'decision': 'READY_FOR_SANDBOX_REVIEW',
 'metrics': {'finding_count': 3,
  'fake_publisher_call_count': 1,
  'duplicate_reused': True}}

## 완료 확인

- Day 4의 1~8차시 결과 파일을 확인했습니다.
- 정상 경로와 가장 중요한 실패 경로를 모두 실행했습니다.
- 외부 쓰기와 자동 메일이 기본값 `false`임을 확인했습니다.
- Codex·Claude Code 결과는 test와 diff를 사람이 검토한 뒤에만 반영합니다.